# Финальный фиксированный LightGBM

Run All запускает одну модель 015_tree: 855 готовых признаков, sqrt-веса, параметры 011_tree tuned и 420 итераций. Screening, search, CV и early stopping отсутствуют. Val используется только после fit. OOF у этого результата нет.

Перед запуском установите окружение uv sync --locked и проверьте пути. Конфигурации 013/014 отменены. Результаты прошлой серии сохраняются.

In [ ]:
import sys
from pathlib import Path

# На Kaggle задайте абсолютный путь к каталогу кода проекта.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if (
    not (PROJECT_ROOT / "pyproject.toml").is_file()
    or not (PROJECT_ROOT / "src" / "__init__.py").is_file()
):
    raise FileNotFoundError("Укажите PROJECT_ROOT: каталог с pyproject.toml и src/")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
DATA_ROOT = PROJECT_ROOT / "data/raw"
if not (DATA_ROOT / "labels.csv").is_file():
    DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "artefacts"
CACHE_ROOT = PROJECT_ROOT / "artefacts/cache/features"  # Только чтение.
FOLD_ROOT = PROJECT_ROOT / "artefacts/splits"  # Сохранённые LR folds.
FAMILY = "tree"  # tree / linear
STAGE = "fixed"  # fixed = одно обучение; cache = только проверка готового кэша
THREADS = "auto"  # -1 = все доступные CPU; также можно задать 8, 10, 12...
SMOKE = False
CONFIG_IDS = (
    ["015_tree"]
    if FAMILY == "tree"
    else [f"{i:03d}_linear" for i in (1, 2, 3, 4, 5, 7)]
)
CONFIGS = [PROJECT_ROOT / "configs" / f"{name}.yaml" for name in CONFIG_IDS]
command = [
    "uv",
    "run",
    "--locked",
    "python",
    "-m",
    "src.run",
    *map(str, CONFIGS),
    "--data-root",
    str(DATA_ROOT),
    "--output-root",
    str(OUTPUT_ROOT),
]
if FAMILY == "tree":
    command += [
        "--stage",
        STAGE,
        "--threads",
        str(THREADS),
        "--cache-root",
        str(CACHE_ROOT),
        "--fold-root",
        str(FOLD_ROOT),
    ]
if SMOKE:
    command.append("--smoke")
print("Конфигурации:", CONFIG_IDS, "этап:", STAGE, "SMOKE:", SMOKE)

Сначала читается полный multi-resolution кэш и проверяются ID, схема и полнота train/val. Несовместимый кэш вызывает ошибку; извлечение сигналов автоматически не запускается. Число потоков auto использует все доступные логические CPU кроме одного, -1 использует все.

Затем один fit на всём train, ровно заданные 420 итераций, и предсказание val. Параметры и число итераций по val не выбираются. Повторное обучение базового/tuned-вариантов не выполняется. Завершённый результат при совпадении настроек используется повторно.

In [ ]:
from src.notebook import run_command

run_command(command, cwd=PROJECT_ROOT)

015_tree сохраняет fixed.txt, fixed.model.json, fixed_predictions.npz (только val), fixed_importance.csv, метрики, разрешённую конфигурацию и provenance. В журнале одна строка с протоколом provided_val_fixed_no_cv; OOF и разброс folds пустые. Для сравнения в 03_eval добавлены результаты без CV. Полное обучение выполняет пользователь; агент его не запускал.

In [ ]:
namespace = OUTPUT_ROOT / "smoke" if SMOKE else OUTPUT_ROOT
summary_code = """
import sys
from pathlib import Path
import polars as pl
from src.evaluation import comparison_table
root = Path(sys.argv[1])
ids = [name for name in sys.argv[2:] if (root / name / 'metrics.json').is_file()]
if ids:
    with pl.Config(tbl_rows=-1, tbl_cols=-1):
        print(comparison_table(root, ids).select('experiment_id','regime','oof_macro_f1','fold_f1_std','val_macro_f1'))
else:
    print('Полных результатов ещё нет. После benchmark измените STAGE вручную для обучения.')
"""
run_command(
    [
        "uv",
        "run",
        "--locked",
        "python",
        "-c",
        summary_code,
        str(namespace),
        *CONFIG_IDS,
    ],
    cwd=PROJECT_ROOT,
)